In [6]:
from pathlib import Path
import pandas as pd
import gc
import shutil

# =========================================================
# 0. 기본 설정
# =========================================================

base_dir = Path("/Users/kimminju/Desktop/FAERS")

start_year = 2023
end_year = 2026

# 최종 DB를 분할 저장할 폴더
output_dir = Path("FAERS_2024_2026_PSSS_DB_parts")

# 이전 실행 결과가 있다면 삭제 후 새로 생성
if output_dir.exists():
    shutil.rmtree(output_dir)

output_dir.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1. 대소문자를 구분하지 않고 폴더/파일 찾는 함수
# =========================================================

def find_case_insensitive(parent: Path, target_name: str):
    """
    parent 폴더 안에서 target_name과 이름이 같은
    폴더 또는 파일을 대소문자 구분 없이 찾는다.
    """
    if not parent.exists():
        return None

    target_lower = target_name.lower()

    for child in parent.iterdir():
        if child.name.lower() == target_lower:
            return child

    return None


def get_faers_file(year, quarter, table_name):
    """
    예:
    2024Q1 DEMO 파일
    /Users/kimminju/Desktop/FAERS/
        faers_ascii_2024Q1/
            ASCII/
                DEMO24Q1.txt
    """

    folder_name = f"faers_ascii_{year}Q{quarter}"
    file_name = f"{table_name}{str(year)[-2:]}Q{quarter}.txt"

    quarter_dir = find_case_insensitive(base_dir, folder_name)

    if quarter_dir is None:
        return None

    ascii_dir = find_case_insensitive(quarter_dir, "ASCII")

    if ascii_dir is None:
        return None

    return find_case_insensitive(ascii_dir, file_name)


# =========================================================
# 2. DEMO 파일만 먼저 합치기
#    같은 caseid 중 최신 primaryid를 선택하기 위함
# =========================================================

demo_list = []
available_quarters = []

for year in range(start_year, end_year + 1):

    for quarter in range(1, 5):

        demo_path = get_faers_file(
            year=year,
            quarter=quarter,
            table_name="DEMO"
        )

        drug_path = get_faers_file(
            year=year,
            quarter=quarter,
            table_name="DRUG"
        )

        reac_path = get_faers_file(
            year=year,
            quarter=quarter,
            table_name="REAC"
        )

        # 세 파일이 모두 있는 분기만 분석
        if demo_path is None or drug_path is None or reac_path is None:
            print(f"[건너뜀] {year}Q{quarter}: DEMO/DRUG/REAC 중 일부 없음")
            continue

        print(f"[DEMO 읽기] {year}Q{quarter}")

        demo = pd.read_csv(
            demo_path,
            sep="$",
            usecols=lambda col: col.lower() in {
                "caseid",
                "primaryid",
                "fda_dt"
            },
            dtype=str,
            encoding="latin1",
            low_memory=False
        )

        # 컬럼 이름 소문자 및 공백 제거
        demo.columns = (
            demo.columns
            .astype(str)
            .str.strip()
            .str.lower()
        )

        for col in ["caseid", "primaryid", "fda_dt"]:
            if col not in demo.columns:
                demo[col] = pd.NA

            demo[col] = demo[col].astype("string").str.strip()

        demo["source_year"] = year
        demo["source_quarter_num"] = quarter
        demo["source_quarter"] = f"{year}Q{quarter}"

        demo_list.append(demo)

        available_quarters.append({
            "year": year,
            "quarter": quarter,
            "quarter_name": f"{year}Q{quarter}",
            "demo_path": demo_path,
            "drug_path": drug_path,
            "reac_path": reac_path
        })

        print(f"  DEMO 행 수: {len(demo):,}")


if not demo_list:
    raise FileNotFoundError(
        "2024~2026년 범위에서 DEMO, DRUG, REAC 파일이 모두 있는 "
        "분기를 찾지 못했습니다."
    )


combined_demo = pd.concat(
    demo_list,
    ignore_index=True
)

del demo_list
gc.collect()

print(f"\n전체 DEMO 행 수: {len(combined_demo):,}")


# =========================================================
# 3. 같은 caseid에서 최신 primaryid 하나만 선택
# =========================================================

combined_demo = combined_demo.dropna(
    subset=["caseid", "primaryid"]
).copy()

combined_demo = combined_demo[
    (combined_demo["caseid"] != "") &
    (combined_demo["primaryid"] != "")
].copy()


# 날짜 정렬용
combined_demo["fda_dt_sort"] = pd.to_datetime(
    combined_demo["fda_dt"],
    format="%Y%m%d",
    errors="coerce"
)


# primaryid는 원본에서는 문자열로 유지하고,
# 정렬할 때만 숫자형 보조 열을 사용
combined_demo["primaryid_sort"] = pd.to_numeric(
    combined_demo["primaryid"],
    errors="coerce"
)


# 최신 FDA_DT 우선
# FDA_DT가 같으면 더 큰 primaryid 우선
dedup_demo = (
    combined_demo
    .sort_values(
        by=[
            "caseid",
            "fda_dt_sort",
            "primaryid_sort"
        ],
        ascending=[
            True,
            False,
            False
        ],
        na_position="last"
    )
    .drop_duplicates(
        subset=["caseid"],
        keep="first"
    )
    [
        [
            "caseid",
            "primaryid",
            "source_year",
            "source_quarter_num",
            "source_quarter"
        ]
    ]
    .reset_index(drop=True)
)


print(f"최신 보고서만 남긴 case 수: {len(dedup_demo):,}")
print(
    "primaryid 고유 개수:",
    f"{dedup_demo['primaryid'].nunique():,}"
)


# DEMO 자체의 동일 primaryid 중복 방지
dedup_demo = dedup_demo.drop_duplicates(
    subset=["primaryid"],
    keep="first"
)


# 메모리 정리
del combined_demo
gc.collect()


# =========================================================
# 4. 분기별로 DRUG와 REAC를 읽고 최신 primaryid만 추출
# =========================================================

total_output_rows = 0
part_number = 1
quarter_summary = []

for quarter_info in available_quarters:

    year = quarter_info["year"]
    quarter = quarter_info["quarter"]
    quarter_name = quarter_info["quarter_name"]

    drug_path = quarter_info["drug_path"]
    reac_path = quarter_info["reac_path"]

    print("\n" + "=" * 70)
    print(f"{quarter_name} 처리 시작")
    print("=" * 70)


    # -----------------------------------------------------
    # 해당 분기에 속한 최신 primaryid 목록
    # -----------------------------------------------------

    demo_q = dedup_demo[
        (dedup_demo["source_year"] == year) &
        (dedup_demo["source_quarter_num"] == quarter)
    ][
        ["caseid", "primaryid"]
    ].copy()

    if demo_q.empty:
        print(f"{quarter_name}: 최신 보고서에 해당하는 사례 없음")
        continue

    valid_primaryids = set(
        demo_q["primaryid"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    print(f"최신 primaryid 수: {len(valid_primaryids):,}")


    # -----------------------------------------------------
    # DRUG 읽기
    # -----------------------------------------------------

    drug = pd.read_csv(
        drug_path,
        sep="$",
        usecols=lambda col: col.lower() in {
            "primaryid",
            "prod_ai",
            "role_cod"
        },
        dtype=str,
        encoding="latin1",
        low_memory=False
    )

    drug.columns = (
        drug.columns
        .astype(str)
        .str.strip()
        .str.lower()
    )

    for col in ["primaryid", "prod_ai", "role_cod"]:
        if col not in drug.columns:
            drug[col] = pd.NA

        drug[col] = drug[col].astype("string").str.strip()


    # role_cod 대문자 통일
    drug["role_cod"] = drug["role_cod"].str.upper()


    # PS 또는 SS만 유지
    drug = drug[
        drug["role_cod"].isin(["PS", "SS"])
    ].copy()


    # 최신 보고서의 primaryid만 유지
    drug = drug[
        drug["primaryid"].isin(valid_primaryids)
    ].copy()


    # prod_ai 결측 및 빈 문자열 제외
    drug = drug[
        drug["prod_ai"].notna() &
        (drug["prod_ai"] != "")
    ].copy()


    # 약물명 앞뒤 공백 정리
    drug["prod_ai"] = (
        drug["prod_ai"]
        .astype("string")
        .str.strip()
    )


    # 같은 primaryid에 동일 약물과 동일 역할이 반복되면 제거
    drug = drug.drop_duplicates(
        subset=[
            "primaryid",
            "prod_ai",
            "role_cod"
        ]
    )


    print(f"PS/SS DRUG 행 수: {len(drug):,}")


    # -----------------------------------------------------
    # REAC 읽기
    # -----------------------------------------------------

    reac = pd.read_csv(
        reac_path,
        sep="$",
        usecols=lambda col: col.lower() in {
            "primaryid",
            "pt"
        },
        dtype=str,
        encoding="latin1",
        low_memory=False
    )

    reac.columns = (
        reac.columns
        .astype(str)
        .str.strip()
        .str.lower()
    )

    for col in ["primaryid", "pt"]:
        if col not in reac.columns:
            reac[col] = pd.NA

        reac[col] = reac[col].astype("string").str.strip()


    # 최신 보고서의 primaryid만 유지
    reac = reac[
        reac["primaryid"].isin(valid_primaryids)
    ].copy()


    # PT 결측 및 빈 문자열 제외
    reac = reac[
        reac["pt"].notna() &
        (reac["pt"] != "")
    ].copy()


    # 같은 primaryid에 같은 PT가 여러 번 있으면 제거
    reac = reac.drop_duplicates(
        subset=[
            "primaryid",
            "pt"
        ]
    )


    print(f"REAC 행 수: {len(reac):,}")


    # -----------------------------------------------------
    # DEMO + DRUG + REAC 병합
    # -----------------------------------------------------

    merged_q = (
        demo_q
        .merge(
            drug,
            on="primaryid",
            how="inner",
            validate="one_to_many"
        )
        .merge(
            reac,
            on="primaryid",
            how="inner",
            validate="many_to_many"
        )
    )


    # 최종 컬럼 순서
    merged_q = merged_q[
        [
            "caseid",
            "primaryid",
            "prod_ai",
            "role_cod",
            "pt"
        ]
    ]


    # 완전히 동일한 drug-event pair 중복 제거
    merged_q = merged_q.drop_duplicates(
        subset=[
            "caseid",
            "primaryid",
            "prod_ai",
            "role_cod",
            "pt"
        ]
    ).reset_index(drop=True)


    # -----------------------------------------------------
    # 분기별 Parquet 저장
    # -----------------------------------------------------

    output_path = (
        output_dir /
        f"FAERS_PSSS_{quarter_name}_part_{part_number:02d}.parquet"
    )

    merged_q.to_parquet(
        output_path,
        index=False
    )

    quarter_summary.append({
        "quarter": quarter_name,
        "latest_primaryids": len(valid_primaryids),
        "drug_rows_ps_ss": len(drug),
        "reac_rows": len(reac),
        "final_pair_rows": len(merged_q),
        "output_file": output_path.name
    })

    total_output_rows += len(merged_q)

    print(f"최종 pair 행 수: {len(merged_q):,}")
    print(f"저장 완료: {output_path}")

    part_number += 1

    del demo_q
    del valid_primaryids
    del drug
    del reac
    del merged_q
    gc.collect()


# =========================================================
# 5. 처리 결과 요약 저장
# =========================================================

summary_df = pd.DataFrame(quarter_summary)

summary_path = output_dir / "FAERS_PSSS_merge_summary.csv"

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 70)
print("전체 작업 완료")
print("=" * 70)

print(f"처리된 분기 수: {len(summary_df):,}")
print(f"최종 전체 drug-event pair 수: {total_output_rows:,}")
print(f"저장 폴더: {output_dir.resolve()}")

display(summary_df)

[DEMO 읽기] 2023Q1
  DEMO 행 수: 432,144
[DEMO 읽기] 2023Q2
  DEMO 행 수: 418,592
[DEMO 읽기] 2023Q3
  DEMO 행 수: 407,522
[DEMO 읽기] 2023Q4
  DEMO 행 수: 415,379
[DEMO 읽기] 2024Q1
  DEMO 행 수: 406,184
[DEMO 읽기] 2024Q2
  DEMO 행 수: 397,119
[DEMO 읽기] 2024Q3
  DEMO 행 수: 405,513
[DEMO 읽기] 2024Q4
  DEMO 행 수: 410,849
[DEMO 읽기] 2025Q1
  DEMO 행 수: 400,514
[DEMO 읽기] 2025Q2
  DEMO 행 수: 393,130
[DEMO 읽기] 2025Q3
  DEMO 행 수: 438,512
[DEMO 읽기] 2025Q4
  DEMO 행 수: 385,288
[DEMO 읽기] 2026Q1
  DEMO 행 수: 397,224
[건너뜀] 2026Q2: DEMO/DRUG/REAC 중 일부 없음
[건너뜀] 2026Q3: DEMO/DRUG/REAC 중 일부 없음
[건너뜀] 2026Q4: DEMO/DRUG/REAC 중 일부 없음

전체 DEMO 행 수: 5,307,970
최신 보고서만 남긴 case 수: 4,566,101
primaryid 고유 개수: 4,566,101

2023Q1 처리 시작
최신 primaryid 수: 356,764
PS/SS DRUG 행 수: 621,305
REAC 행 수: 1,052,644
최종 pair 행 수: 2,815,128
저장 완료: FAERS_2024_2026_PSSS_DB_parts/FAERS_PSSS_2023Q1_part_01.parquet

2023Q2 처리 시작
최신 primaryid 수: 346,239
PS/SS DRUG 행 수: 611,391
REAC 행 수: 1,062,656
최종 pair 행 수: 3,194,797
저장 완료: FAERS_2024_2026_PSSS_DB_parts/FAERS_PSSS

,quarter,latest_primaryids,drug_rows_ps_ss,reac_rows,final_pair_rows,output_file
0,2023Q1,356764,621305,1052644,2815128,FAERS_PSSS_2023Q1_part_01.parquet
1,2023Q2,346239,611391,1062656,3194797,FAERS_PSSS_2023Q2_part_02.parquet
2,2023Q3,349706,618308,1029657,2823083,FAERS_PSSS_2023Q3_part_03.parquet
3,2023Q4,341931,635650,1043918,3054678,FAERS_PSSS_2023Q4_part_04.parquet
4,2024Q1,337925,603393,1022798,3260790,FAERS_PSSS_2024Q1_part_05.parquet
5,2024Q2,332826,607300,1040597,4011291,FAERS_PSSS_2024Q2_part_06.parquet
6,2024Q3,346607,630264,1071540,4290992,FAERS_PSSS_2024Q3_part_07.parquet
7,2024Q4,337675,628337,1051903,4303954,FAERS_PSSS_2024Q4_part_08.parquet
8,2025Q1,329175,611482,1030234,4245009,FAERS_PSSS_2025Q1_part_09.parquet
9,2025Q2,332036,592637,1012393,3542376,FAERS_PSSS_2025Q2_part_10.parquet


In [7]:
from pathlib import Path
import pandas as pd

output_dir = Path("FAERS_2024_2026_PSSS_DB_parts")

parquet_files = sorted(
    output_dir.glob("FAERS_PSSS_*.parquet")
)

print(f"불러올 파일 수: {len(parquet_files)}")

final_db = pd.concat(
    [
        pd.read_parquet(file)
        for file in parquet_files
    ],
    ignore_index=True
)

print("최종 DB shape:", final_db.shape)
print(final_db.head())

불러올 파일 수: 13
최종 DB shape: (50906205, 5)
     caseid  primaryid             prod_ai role_cod                     pt
0  10026447  100264472           RITUXIMAB       PS         Cardiac arrest
1  10026447  100264472           RITUXIMAB       PS  Myocardial infarction
2  10026447  100264472  METHYLPREDNISOLONE       SS         Cardiac arrest
3  10026447  100264472  METHYLPREDNISOLONE       SS  Myocardial infarction
4  10026447  100264472        PREDNISOLONE       SS         Cardiac arrest


In [8]:
final_db.to_csv(
    "FAERS_2023_2026_PSSS_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("최종 CSV 저장 완료")

최종 CSV 저장 완료
